In [0]:
basePath = "/Volumes/workspace/default/dataflowmed"
silverPath = f"{basePath}/silver"
goldPath = f"{basePath}/gold"

def registerSilverView(tableName, viewName=None):
    viewName = viewName or tableName
    spark.read.format("delta").load(f"{silverPath}/{tableName}").createOrReplaceTempView(viewName)

registerSilverView("olist_customers_dataset", "silverCustomers")
registerSilverView("olist_products_dataset", "silverProducts")
registerSilverView("product_category_name_translation", "silverCategoryTranslation")
registerSilverView("olist_sellers_dataset", "silverSellers")
registerSilverView("olist_orders_dataset", "silverOrders")
registerSilverView("olist_order_items_dataset", "silverOrderItems")
registerSilverView("olist_order_payments_dataset", "silverPayments")

In [0]:
%sql
CREATE OR REPLACE TABLE delta.`/Volumes/workspace/default/dataflowmed/gold/dimCustomers`
AS
SELECT
customer_id,
customer_unique_id,
customer_city,
customer_state,
customer_zip_code_prefix
FROM silverCustomers
WHERE isCurrent = true

num_affected_rows,num_inserted_rows


In [0]:
%sql
CREATE OR REPLACE TABLE delta.`/Volumes/workspace/default/dataflowmed/gold/dimProducts`
AS
SELECT
    p.product_id,
    p.product_category_name,
    COALESCE(c.product_category_name_english, p.product_category_name) AS product_category,
    p.product_weight_g
FROM silverProducts p
LEFT JOIN silverCategoryTranslation c
    ON p.product_category_name = c.product_category_name

num_affected_rows,num_inserted_rows


In [0]:
%sql
CREATE OR REPLACE TABLE delta.`/Volumes/workspace/default/dataflowmed/gold/dimSellers`
AS
SELECT
seller_id,
seller_city,
seller_state
FROM silverSellers

num_affected_rows,num_inserted_rows


In [0]:
%sql
CREATE OR REPLACE TABLE delta.`/Volumes/workspace/default/dataflowmed/gold/factOrderItems`
AS
SELECT
oi.order_id,
oi.product_id,
oi.seller_id,
o.customer_id,
o.order_status,
o.order_purchase_timestamp,
oi.price,
oi.freight_value,
(oi.price + oi.freight_value) AS total_item_value
FROM silverOrderItems oi
INNER JOIN silverOrders o
ON oi.order_id = o.order_id

num_affected_rows,num_inserted_rows


In [0]:
%sql
CREATE OR REPLACE TABLE delta.`/Volumes/workspace/default/dataflowmed/gold/factPayments`
AS
WITH rankedPayments AS (
SELECT
order_id,
payment_value,
payment_type,
ROW_NUMBER() OVER (PARTITION BY order_id ORDER BY payment_value DESC) AS paymentRank
FROM silverPayments
)
SELECT
order_id,
SUM(payment_value) AS total_payment_value,
MAX(CASE WHEN paymentRank = 1 THEN payment_type END) AS primary_payment_type
FROM rankedPayments
GROUP BY order_id

num_affected_rows,num_inserted_rows


In [0]:
display(dbutils.fs.ls(goldPath))

path,name,size,modificationTime
dbfs:/Volumes/workspace/default/dataflowmed/gold/dimCustomers/,dimCustomers/,0,1784472365183
dbfs:/Volumes/workspace/default/dataflowmed/gold/dimProducts/,dimProducts/,0,1784472365183
dbfs:/Volumes/workspace/default/dataflowmed/gold/dimSellers/,dimSellers/,0,1784472365183
dbfs:/Volumes/workspace/default/dataflowmed/gold/factOrderItems/,factOrderItems/,0,1784472365183
dbfs:/Volumes/workspace/default/dataflowmed/gold/factPayments/,factPayments/,0,1784472365183
